In [1]:
# Autoload modules
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')
from fcc import MiyazawaJerniganInteraction, Peptide, ProteinFoldingProblem, PenaltyParameters
from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_algorithms.optimizers import COBYLA, SLSQP
from qiskit_algorithms.minimum_eigensolvers import SamplingVQE
from qiskit.primitives import Sampler
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import Statevector

/Users/ruihaoli/protein-folding-qc/notebooks/../fcc/fcc_qubit_op_builder.py:259: SyntaxWarning: invalid escape sequence '\l'
  """


In [3]:
def build_pf(main_seq: str, energy_matrix_file: str = "mj_matrix"):
    """Builds the protein folding problem for the given sequence."""

    mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file)
    # print(mj_interaction.calculate_energy_matrix(main_seq))

    penalty_back = 50
    penalty_redun = 50
    penalty_olap = 50

    penalty_terms = PenaltyParameters(penalty_back, penalty_redun, penalty_olap)

    peptide = Peptide(main_seq)

    protein_folding_problem = ProteinFoldingProblem(
        peptide, mj_interaction, penalty_terms
    )

    return protein_folding_problem

/Users/ruihaoli/protein-folding-qc/notebooks/../fcc/fcc_qubit_op_builder.py:259: SyntaxWarning: invalid escape sequence '\l'
  """


In [4]:
# main_seq = "LHPGAGK" # Zika
main_seq = "LFLF" # 6MVZ ("LFLF")
protein_folding_problem = build_pf(main_seq)

qubit_op = protein_folding_problem.qubit_op()
print("Number of qubits: ", qubit_op.num_qubits)
print("Number of terms: ", len(qubit_op))
print(qubit_op)

Highest degree of the polynomial for 0 and 3: 6
Number of qubits:  9
Number of terms:  131
SparsePauliOp(['IIIIIIIII', 'IIIZZIIII', 'IIIZIZZII', 'IIIIZZZII', 'IIIIIIZZI', 'IIIZZIZZI', 'IIIZIZIZI', 'IIIIZZIZI', 'IIIIIZIIZ', 'IIIZZZIIZ', 'IIIZIIZIZ', 'IIIIZIZIZ', 'IIIIIZZZZ', 'IIIZZZZZZ', 'IIIZIIIZZ', 'IIIIZIIZZ', 'IIIIIZZII', 'IIIZZZZII', 'IIIIIIIZI', 'IIIIIZIII', 'IIIZIZIII', 'IIIIZZIII', 'IIIZZZIII', 'IIIIIIZII', 'IIIZIIZII', 'IIIIZIZII', 'IIIZZIZII', 'IIIIIZIZI', 'IIIZZZIZI', 'IIIZIIZZI', 'IIIZIIIII', 'IIIIZIIII', 'IIIIIIIZZ', 'IIIIIZIZZ', 'IIIZZZIZZ', 'IIIIIIZZZ', 'IIIZZIZZZ', 'IIIZIZZZZ', 'IIIIZZZZZ', 'IIIIZZIIZ', 'IIIIZIZZI', 'IIIZIIIZI', 'IIIIZIIZI', 'IIIZZIIZI', 'IIIIIZZZI', 'IIIZIZZZI', 'IIIIZZZZI', 'IIIZZZZZI', 'IIIIIIIIZ', 'IIIZIIIIZ', 'IIIIZIIIZ', 'IIIIIIZIZ', 'IIIZZIZIZ', 'IIIZIZZIZ', 'IIIIZZZIZ', 'IIIIZZIZZ', 'IIIIZIZZZ', 'IIIZIZIZZ', 'IIIZZIIZZ', 'IIIZIIZZZ', 'IIIZIZIIZ', 'IIIZZIIIZ', 'IIIZZZZIZ', 'IIIIIZZIZ', 'IIZIIIIII', 'IIZIIIIZI', 'IIZIIIIZZ', 'IZIIIIIII', 'IZIIIIIZI

Below we incorporate the sampling VQE approach, where we store all the configurations sampled during the VQE optimization. In the end, we choose the top 100 configurations as the final result and save them in a file.

In [ ]:
#TODO: Finish this